In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW  = Path('../data/raw')
PREP = Path('../data/prep')

def reduce_memory_usage(df, name=''):
    start = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        t = str(df[col].dtype)
        if t.startswith('int'):
            for typ in [np.int8, np.int16, np.int32]:
                info = np.iinfo(typ)
                if df[col].min() > info.min and df[col].max() < info.max:
                    df[col] = df[col].astype(typ)
                    break
        elif t.startswith('float'):
            df[col] = df[col].astype(np.float32)
    end = df.memory_usage().sum() / 1024**2
    print(f'[{name}] {start:.1f}MB → {end:.1f}MB ({100*(start-end)/start:.1f}% 감소)')
    return df

print("=== 원본 데이터 로드 ===")
orders        = pd.read_csv(RAW / 'orders.csv',
                            usecols=['order_id','user_id','eval_set',
                                     'order_dow','order_hour_of_day','days_since_prior_order'])
prior         = pd.read_csv(RAW / 'order_products__prior.csv',
                            usecols=['order_id','product_id'])
train_labels  = pd.read_csv(RAW / 'order_products__train.csv',
                            usecols=['order_id','product_id'])

orders        = reduce_memory_usage(orders,       'orders')
prior         = reduce_memory_usage(prior,        'prior')
train_labels  = reduce_memory_usage(train_labels, 'train_labels')
print("로드 완료")

In [ ]:
# ================================================================
# Step 1: 후보 집합(Candidates) 생성 — 중복/누수 없는 핵심 단계
# ================================================================
# [기존 방법의 문제]
#   orders(전체) ⋈ order_products__prior → prior 주문 수만큼 행 폭발
#   reordered(prior) 를 label로 쓰면 up_reorder_rate와 거의 동일 → leakage
#
# [올바른 방법]
#   대상: train 유저만 (eval_set == 'train')
#   후보: 해당 유저가 prior 에서 구매한 상품 목록 (user, product) 유니크
#   label: train 주문에 실제로 담겼으면 1, 아니면 0

print("=== Step 1: 후보 집합 생성 ===")

# train 유저의 마지막 주문 정보
train_orders = orders[orders['eval_set'] == 'train'][
    ['user_id', 'order_id', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
].copy()

# prior 주문 → user_id 붙이기
prior_orders = orders[orders['eval_set'] == 'prior'][['order_id', 'user_id']]
prior_with_user = prior.merge(prior_orders, on='order_id', how='inner')

# 후보: train 유저 × 과거 구매 상품 (유니크)
train_user_ids = train_orders['user_id'].unique()
candidates = (prior_with_user[prior_with_user['user_id'].isin(train_user_ids)]
              [['user_id', 'product_id']]
              .drop_duplicates())

print(f"후보 (user, product) 쌍: {len(candidates):,}행")

# ================================================================
# Step 2: label 붙이기 — train 주문에 실제 담긴 상품만 1
# ================================================================
# train_labels에 user_id 연결
train_label_with_user = train_labels.merge(
    train_orders[['user_id', 'order_id']], on='order_id', how='inner'
)[['user_id', 'product_id']].copy()
train_label_with_user['label'] = 1

candidates = candidates.merge(train_label_with_user, on=['user_id', 'product_id'], how='left')
candidates['label'] = candidates['label'].fillna(0).astype(np.int8)

print(f"label=1 (재구매): {candidates['label'].sum():,}")
print(f"label=0 (미구매): {(candidates['label']==0).sum():,}")
print(f"재구매율: {candidates['label'].mean():.3f}")

# train 주문 정보 병합 (주문 시간대, 요일 등 — prior 정보 아님)
candidates = candidates.merge(
    train_orders[['user_id', 'order_id', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']],
    on='user_id', how='left'
)

del prior_orders, prior_with_user, train_label_with_user
print("Step 1 완료")

In [ ]:
# ================================================================
# Step 2: 피처 병합 (prior 기반 피처만 사용 — leakage 없음)
# ================================================================
print("=== Step 2: 피처 병합 ===")

# 유저-상품 피처 (prior에서 계산된 것)
up_feat = reduce_memory_usage(
    pd.read_csv(PREP / 'user_product_features_final.csv'), 'up_features')
candidates = candidates.merge(up_feat, on=['user_id', 'product_id'], how='left')
del up_feat

# 유저 피처
u_feat = reduce_memory_usage(
    pd.read_csv(PREP / 'user_features.csv'), 'user_features')
candidates = candidates.merge(u_feat, on='user_id', how='left')
del u_feat

# 상품 피처 (aisle_id, department_id 중복 주의)
p_feat = reduce_memory_usage(
    pd.read_csv(PREP / 'prod_features.csv'), 'prod_features')
# aisle_id, department_id는 user_product_features에 이미 있으므로 제거
p_feat = p_feat.drop(columns=['aisle_id', 'department_id'], errors='ignore')
candidates = candidates.merge(p_feat, on='product_id', how='left')
del p_feat

# 중복 컬럼 제거 (_x/_y 접미사)
dup_cols = [c for c in candidates.columns if c.endswith(('_x', '_y'))]
if dup_cols:
    print(f"중복 컬럼 제거: {dup_cols}")
    candidates.drop(columns=dup_cols, inplace=True)

candidates = reduce_memory_usage(candidates, 'final')
print(f"최종 테이블 크기: {candidates.shape}")
print(candidates.dtypes)

In [ ]:
# ================================================================
# Step 3: 무결성 검증 및 저장
# ================================================================
print("=== Step 3: 무결성 검증 ===")

# (user_id, product_id) 중복 없어야 함
dup_count = candidates.duplicated(['user_id', 'product_id']).sum()
print(f"(user, product) 중복 행: {dup_count}  ← 0이어야 정상")

# 결측치 확인
missing = candidates.isnull().sum()
missing = missing[missing > 0]
if len(missing):
    print(f"\n결측치 있는 컬럼:\n{missing}")
else:
    print("결측치 없음")

# label 분포
print(f"\nlabel 분포:\n{candidates['label'].value_counts()}")
print(f"재구매율: {candidates['label'].mean():.4f}  ← 대회 기준 약 0.59")

# 저장
out_path = PREP / 'k-pick_total_v3.csv'
candidates.to_csv(out_path, index=False, float_format='%.4f')
print(f"\n저장 완료: {out_path}")
print(f"행: {len(candidates):,} / 컬럼: {candidates.shape[1]}")